# ML3

## 1. Ответы на вопросы 
1. Leave-one-out - это такой случай кросс валидации, при котором берется только один объект для тестовой части, а все остальные для тренировочной. Т.е., если всего n, то n-1 - тренировочные и 1 тестовый и так n раз. Итоговая метрика получается усреденением всех. Его плюсы в том, что используется максимальное количество данных для обучения, это уменьшает переобучение. Минус в том, что это очень медленно, что бессмысленно для больших датасетов

2. Grid search - принимает сколько угодно параметров и считает точность для каждого, и выбирает лучший. Random search - берет случайные значения и ищет лучший. А байесовская оптимизация делает несколько случайных измерений и по ним определяет, куда шагать дальше для улучшения точности и процесс повторяется.

3. Классификация методов отбора признаков: фильтры - определяют значимость признаков до обучения модели, на основе статистики данных, они вычисляют статистическую меру (коэффициент корреляции, хи-квадрат, дисперсию) для каждого признака и отбирают лучшие по пороговому значению, например коэффицент пирсона или хи-квадрат. 2) Встроенные методы (Embedded Methods) отбор признаков происходит в процессе обучения модели. Алгоритм обучения автоматически определяет важность признаков как часть своей оптимизации. Например,  Lasso-регуляризация, деревья решений, случайные леса. 3) 3. Оберточные методы (Wrapper Methods). Используют производительность модели как критерий для отбора признаков. Перебирают подмножества признаков, каждый раз обучая модель и оценивая её качество. Примеры: Recursive Feature Elimination (RFE), Forward/Backward Selection. 4) Методы на основе важности признаков (Feature Importance Methods). Анализируют уже обученную модель, чтобы понять вклад каждого признака. Permutation Importance, SHAP, LIME.

Коэффицент пирсона использует корреляцию между признаками и исключает одну из сильно зависимых друг от друга.

Хи-квадрат определяет зависимоть целевой переменной и признаков, если они 0, то признак исключается.

Лассо - добавляет штраф за абсолютные значения коэффицентов. При его увеличении некоторые становятся равными 0

Permutation Importance - сначала считается важность признаков. После их перемешивают и заново считают.

SHAP основан на теории кооперативных игр и оценивает вклад каждого признака в предсказание модели относительно базового значения.

## 2. Введение

In [172]:
import pandas as pd
import numpy as np
import re

In [173]:
df = pd.read_json('data/train.json')
df

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium
10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,5ba989232d0489da1b5f2c45f6688adc,[https://photos.renthop.com/2/7211212_1ed4542e...,3000,792 Metropolitan Avenue,medium
15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, Fitness Center, Laundry in...",40.7439,7225292,-73.9743,2c3b41f588fbb5234d8a1e885a436cfa,[https://photos.renthop.com/2/7225292_901f1984...,2795,340 East 34th Street,low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124000,1.0,3,92bbbf38baadfde0576fc496bd41749c,2016-04-05 03:58:33,There is 700 square feet of recently renovated...,W 171 Street,"[Elevator, Dishwasher, Hardwood Floors]",40.8433,6824800,-73.9396,a61e21da3ba18c7a3d54cfdcc247e1f8,[https://photos.renthop.com/2/6824800_0682be16...,2800,620 W 171 Street,low
124002,1.0,2,5565db9b7cba3603834c4aa6f2950960,2016-04-02 02:25:31,"2 bedroom apartment with updated kitchen, rece...",Broadway,"[Common Outdoor Space, Cats Allowed, Dogs Allo...",40.8198,6813268,-73.9578,8f90e5e10e8a2d7cf997f016d89230eb,[https://photos.renthop.com/2/6813268_1e6fcc32...,2395,3333 Broadway,medium
124004,1.0,1,67997a128056ee1ed7d046bbb856e3c7,2016-04-26 05:42:03,No Brokers Fee * Never Lived 1 Bedroom 1 Bathr...,210 Brighton 15th St,"[Dining Room, Elevator, Pre-War, Laundry in Bu...",40.5765,6927093,-73.9554,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/6927093_93a52104...,1850,210 Brighton 15th St,medium
124008,1.0,2,3c0574a740154806c18bdf1fddd3d966,2016-04-19 02:47:33,Wonderful Bright Chelsea 2 Bedroom apartment o...,West 21st Street,"[Pre-War, Laundry in Unit, Dishwasher, No Fee,...",40.7448,6892816,-74.0017,c3cd45f4381ac371507090e9ffabea80,[https://photos.renthop.com/2/6892816_1a8d087a...,4195,350 West 21st Street,medium


In [174]:
encodes = {'low': 1, 'medium': 2,'high': 3}
df['interest_level'] = df['interest_level'].map(encodes)

In [175]:
top_20_features = ['Elevator', 'HardwoodFloors', 'CatsAllowed', 'DogsAllowed', 
                    'Doorman', 'Dishwasher', 'NoFee', 'LaundryinBuilding',
                    'FitnessCenter', 'Pre-War', 'LaundryinUnit', 'RoofDeck',
                    'OutdoorSpace', 'DiningRoom', 'HighSpeedInternet', 'Balcony',
                   'SwimmingPool', 'NewConstruction', 'Terrace'
]

df['features'] = df['features'].apply(
    lambda lst: [re.sub(r'[\s\'"]', '', f) for f in lst if isinstance(f, str)]
)

for feature in top_20_features:
    df[feature] = df['features'].apply(lambda x: 1 if feature in x else 0)


df

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,...,Pre-War,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,NewConstruction,Terrace
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[DiningRoom, Pre-War, LaundryinBuilding, Dishw...",40.7108,7170325,-73.9539,...,1,0,0,0,1,0,0,0,0,0
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, LaundryinBuilding, Dishwas...",40.7513,7092344,-73.9722,...,0,0,0,0,0,0,0,0,0,0
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, LaundryinBuilding, Laundry...",40.7575,7158677,-73.9625,...,0,1,0,0,0,0,0,0,0,0
10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,...,0,0,0,0,0,0,0,0,0,0
15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, FitnessCenter, LaundryinBu...",40.7439,7225292,-73.9743,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124000,1.0,3,92bbbf38baadfde0576fc496bd41749c,2016-04-05 03:58:33,There is 700 square feet of recently renovated...,W 171 Street,"[Elevator, Dishwasher, HardwoodFloors]",40.8433,6824800,-73.9396,...,0,0,0,0,0,0,0,0,0,0
124002,1.0,2,5565db9b7cba3603834c4aa6f2950960,2016-04-02 02:25:31,"2 bedroom apartment with updated kitchen, rece...",Broadway,"[CommonOutdoorSpace, CatsAllowed, DogsAllowed,...",40.8198,6813268,-73.9578,...,0,0,0,0,0,0,0,0,0,0
124004,1.0,1,67997a128056ee1ed7d046bbb856e3c7,2016-04-26 05:42:03,No Brokers Fee * Never Lived 1 Bedroom 1 Bathr...,210 Brighton 15th St,"[DiningRoom, Elevator, Pre-War, LaundryinBuild...",40.5765,6927093,-73.9554,...,1,1,0,0,1,0,0,0,0,0
124008,1.0,2,3c0574a740154806c18bdf1fddd3d966,2016-04-19 02:47:33,Wonderful Bright Chelsea 2 Bedroom apartment o...,West 21st Street,"[Pre-War, LaundryinUnit, Dishwasher, NoFee, Ou...",40.7448,6892816,-74.0017,...,1,1,0,1,0,0,0,0,0,0


In [176]:
final_feature_list = ['bathrooms', 'bedrooms', 'interest_level'] + top_20_features
X = df[final_feature_list].values
y = df['price'].values
df['created'] = pd.to_datetime(df['created']).values
dates = df['created'].values
groups = df['interest_level'].values
X

array([[1., 1., 2., ..., 0., 0., 0.],
       [1., 2., 1., ..., 0., 0., 0.],
       [1., 2., 2., ..., 0., 0., 0.],
       ...,
       [1., 1., 2., ..., 0., 0., 0.],
       [1., 2., 2., ..., 0., 0., 0.],
       [1., 3., 3., ..., 0., 0., 0.]])

3. Обычные схемы разбиения

In [177]:
def my_train_test_split(X, y, test_size=0.2, seed=21):
    np.random.seed(seed)
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    len_test = int(test_size * len(X))
    train_ind = indices[len_test:]
    test_ind = indices[:len_test]
    return X[train_ind], X[test_ind], y[train_ind], y[test_ind]

def my_train_val_test_split(X, y, test_size=0.2, val_size=0.2, seed=21):
    np.random.seed(seed)
    indices = np.arange(len(X))
    len_test = int(test_size * len(X))
    len_val = int(val_size * len(X))
    train_ind = indices[len_test + len_val:]
    validation_ind = indices[len_test: len_test + len_val]
    test_ind = indices[:len_test]
    return X[train_ind], X[validation_ind], X[test_ind], y[train_ind], y[validation_ind], y[test_ind]


def my_train_test_split_date(X, y, dates, date_split):
    train_ind = np.where(dates < date_split)[0]
    test_ind = np.where(dates >= date_split)[0]
    return X[train_ind], X[test_ind], y[train_ind], y[test_ind]

def my_train_val_test_split_date(X, y, dates, validation_date, test_date):
    train_ind = np.where(dates < validation_date)[0]
    validation_ind = np.where((dates >= validation_date) & (dates < test_date))[0]
    test_ind = np.where(dates >= test_date)[0]
    return X[train_ind], X[validation_ind], X[test_ind], y[train_ind], y[validation_ind], y[test_ind]



Детерминированное - значит, что разбиение всегда будет одинаковым при повторном запуске кода.

## 4.Методы перектрестной проверки 

In [178]:
def my_kfold(n, k=5, random_state=21):
    np.random.seed(random_state)
    ind = np.arange(n)
    np.random.shuffle(ind)
    fold_sizes = np.full(k, n // k)
    fold_sizes[:n % k] += 1
    current = 0
    folds = []
    for fold_size in fold_sizes:
        test_ind = ind[current:current + fold_size]
        train_ind = np.setdiff1d(ind, test_ind)
        folds.append((train_ind, test_ind))
        current += fold_size
    
    return folds


def my_group_kfold(group_field, k=3, random_state=21):
    np.random.seed(random_state)
    unique_groups = np.unique(group_field)
    np.random.shuffle(unique_groups)
    n = len(unique_groups)
    fold_sizes = np.full(k, n // k)
    fold_sizes[:n % k] += 1
    current = 0
    folds = []
    for fold_size in fold_sizes:
        test_groups = unique_groups[current:current + fold_size]
        current += fold_size
        test_ind = np.where(np.isin(group_field, test_groups))[0]
        train_ind = np.where(~np.isin(group_field, test_groups))[0]
        folds.append((train_ind, test_ind))

    return folds


def my_stratified_kfold(stratify_field, k=5, random_state=21):
    np.random.seed(random_state)
    stratify_field = np.asarray(stratify_field)
    n = len(stratify_field)
    classes = np.unique(stratify_field)
    folds = [[] for _ in range(k)]
    for cls in classes:
        ind = np.where(stratify_field == cls)[0]
        np.random.shuffle(ind)
        parts = np.array_split(ind, k)
        for i in range(k):
            folds[i].extend(parts[i])

    res = []
    all_ind = np.arange(n)
    for i in range(k):
        test_ind = np.array(folds[i])
        train_ind = np.setdiff1d(all_ind, test_ind)
        res.append((train_ind, test_ind))

    return res

def my_time_series_split(date_field, k=5):
    date_field = np.asarray(date_field)
    indices = np.argsort(date_field)
    n = len(indices)
    fold_size = n // (k + 1)
    folds = []
    for i in range(1, k + 1):
        train_end = i * fold_size
        test_end = (i + 1) * fold_size
        train_idx = indices[:train_end]
        test_idx = indices[train_end:test_end]
        folds.append((train_idx, test_idx))

    return folds

5. Сравнение KFOLD

In [179]:
from sklearn.model_selection import train_test_split, KFold, GroupKFold, TimeSeriesSplit

In [180]:
folds_my = my_kfold(len(X), k=5)

kf = KFold(n_splits=5, shuffle=True, random_state=21)
folds_sk = list(kf.split(X))

for i, ((tr_my, te_my), (tr_sk, te_sk)) in enumerate(zip(folds_my, folds_sk)):
    print(f"Fold {i}:")
    print("  Train sizes: My:", len(tr_my), "Sk:", len(tr_sk))
    print("  Test sizes:  My:", len(te_my), "Sk:", len(te_sk))

Fold 0:
  Train sizes: My: 39481 Sk: 39481
  Test sizes:  My: 9871 Sk: 9871
Fold 1:
  Train sizes: My: 39481 Sk: 39481
  Test sizes:  My: 9871 Sk: 9871
Fold 2:
  Train sizes: My: 39482 Sk: 39482
  Test sizes:  My: 9870 Sk: 9870
Fold 3:
  Train sizes: My: 39482 Sk: 39482
  Test sizes:  My: 9870 Sk: 9870
Fold 4:
  Train sizes: My: 39482 Sk: 39482
  Test sizes:  My: 9870 Sk: 9870


In [181]:
folds_group_my = my_group_kfold(groups, k=3)

gkf = GroupKFold(n_splits=3)
folds_group_sk = list(gkf.split(X, y, groups))

for i, ((tr_my, te_my), (tr_sk, te_sk)) in enumerate(zip(folds_group_my, folds_group_sk)):
    print(f"Fold {i}:")
    print("  Train groups My:", len(np.unique(groups[tr_my])), "Sk:", len(np.unique(groups[tr_sk])))
    print("  Test groups  My:", len(np.unique(groups[te_my])), "Sk:", len(np.unique(groups[te_sk])))


Fold 0:
  Train groups My: 2 Sk: 2
  Test groups  My: 1 Sk: 1
Fold 1:
  Train groups My: 2 Sk: 2
  Test groups  My: 1 Sk: 1
Fold 2:
  Train groups My: 2 Sk: 2
  Test groups  My: 1 Sk: 1


In [182]:
folds_time_my = my_time_series_split(dates, k=5)

tscv = TimeSeriesSplit(n_splits=5)
folds_time_sk = list(tscv.split(X))

for i, ((tr_my, te_my), (tr_sk, te_sk)) in enumerate(zip(folds_time_my, folds_time_sk)):
    print(f"Fold {i}:")
    print("  Train size My:", len(tr_my), "Sk:", len(tr_sk))
    print("  Test size  My:", len(te_my), "Sk:", len(te_sk))


Fold 0:
  Train size My: 8225 Sk: 8227
  Test size  My: 8225 Sk: 8225
Fold 1:
  Train size My: 16450 Sk: 16452
  Test size  My: 8225 Sk: 8225
Fold 2:
  Train size My: 24675 Sk: 24677
  Test size  My: 8225 Sk: 8225
Fold 3:
  Train size My: 32900 Sk: 32902
  Test size  My: 8225 Sk: 8225
Fold 4:
  Train size My: 41125 Sk: 41127
  Test size  My: 8225 Sk: 8225


In [183]:
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score


def r2_by_folds(X, y, folds, name=""):
    print(f"\n{name}")
    r2_scores = []

    for i, (tr, te) in enumerate(folds):
        X_tr, X_te = X[tr], X[te]
        y_tr, y_te = y[tr], y[te]

        scaler = MinMaxScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_te = scaler.transform(X_te)

        model = ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000, random_state=21)
        model.fit(X_tr, y_tr)

        y_pred = model.predict(X_te)
        r2 = r2_score(y_te, y_pred)

        r2_scores.append(r2)
        print(f"Fold {i}: R2 = {r2:.6f}")

    print("Mean R2:", np.mean(r2_scores))
    return r2_scores

r2_my_kfold = r2_by_folds(X, y, folds_my, "My KFold")
r2_my_group = r2_by_folds(X, y, folds_group_my, "My GroupKFold")
r2_my_time = r2_by_folds(X, y, folds_time_my, "My TimeSeriesSplit")




My KFold
Fold 0: R2 = 0.281069
Fold 1: R2 = 0.008987
Fold 2: R2 = 0.001723
Fold 3: R2 = 0.267864
Fold 4: R2 = 0.352358
Mean R2: 0.18240025717131525

My GroupKFold
Fold 0: R2 = 0.002678
Fold 1: R2 = -0.020151
Fold 2: R2 = 0.188314
Mean R2: 0.056947075006423674

My TimeSeriesSplit
Fold 0: R2 = 0.400830
Fold 1: R2 = 0.015472
Fold 2: R2 = 0.019767
Fold 3: R2 = 0.329180
Fold 4: R2 = 0.001406
Mean R2: 0.15333081647813857


In [184]:
y_strat = pd.qcut(y, q=5, labels=False)
skf = my_stratified_kfold(y_strat,  random_state=21)

r2_my_time = r2_by_folds(X, y_strat, skf, "My Stratify KFOLD")


My Stratify KFOLD
Fold 0: R2 = 0.530198
Fold 1: R2 = 0.522539
Fold 2: R2 = 0.530634
Fold 3: R2 = 0.525147
Fold 4: R2 = 0.522892
Mean R2: 0.5262821171227021


Подойдет любая, особенно обычный и stratify, но его нужно забинить перед использованием

6. Feature Selection

In [185]:
from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler

In [186]:
lasso = Lasso(alpha=0.01, max_iter=10000)

In [187]:
X_train, X_val, X_test, y_train, y_val, y_test = my_train_val_test_split(X, y)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

lasso.fit(X_train_scaled, y_train)
y_pred_train = lasso.predict(X_train_scaled)
y_pred_val = lasso.predict(X_val_scaled)
y_pred_test = lasso.predict(X_test_scaled)

mae_train = mean_absolute_error(y_train, y_pred_train)
mae_val = mean_absolute_error(y_val, y_pred_val)
mae_test = mean_absolute_error(y_test, y_pred_test)

rmse_train = root_mean_squared_error(y_train, y_pred_train)
rmse_val = root_mean_squared_error(y_val, y_pred_val)
rmse_test = root_mean_squared_error(y_test, y_pred_test)

r2_train = r2_score(y_train, y_pred_train)
r2_val = r2_score(y_val, y_pred_val)
r2_test = r2_score(y_test, y_pred_test)

mae_table = pd.DataFrame(columns=['model','train', 'valid','test'])
rmse_table = pd.DataFrame(columns=['model','train', 'valid','test'])
r2_table = pd.DataFrame(columns=['model','train', 'valid','test'])


mae_table.loc[1] = ('Lasso MinMaxScaler', mae_train, mae_val, mae_test)
rmse_table.loc[1] = ('Lasso MinMaxScaler', rmse_train, rmse_val, rmse_test)
r2_table.loc[1] = ('Lasso MinMaxScaler', r2_train, r2_val, r2_test)

feature_importance = pd.DataFrame({
        'feature': final_feature_list,
        'abs_coef': np.abs(lasso.coef_)
    }).sort_values('abs_coef', ascending=False)
top10 = feature_importance.head(10)
top10

,feature,abs_coef
0,bathrooms,24172.038488
1,bedrooms,2891.225826
7,Doorman,1078.474560
2,interest_level,969.570509
21,Terrace,622.011368
13,LaundryinUnit,538.687328
10,LaundryinBuilding,343.869720
6,DogsAllowed,334.672626
18,Balcony,252.861559
17,HighSpeedInternet,250.617145


In [188]:
cols10 = top10.index
X10 = X[cols10]
X_tr, X_val, X_te, y_tr, y_val, y_te = my_train_val_test_split(
    X10, y, 0.2, 0.2, seed=21
)
scaler = MinMaxScaler()
X_tr = scaler.fit_transform(X_tr)
X_val = scaler.transform(X_val)
X_te = scaler.transform(X_te)

lasso_10 = Lasso(alpha=0.01, max_iter=10000)
lasso_10.fit(X_tr, y_tr)

y_pred_train = lasso_10.predict(X_tr)
y_pred_val = lasso_10.predict(X_val)
y_pred_test = lasso_10.predict(X_te)

mae_train = mean_absolute_error(y_tr, y_pred_train)
mae_val = mean_absolute_error(y_val, y_pred_val)
mae_test = mean_absolute_error(y_te, y_pred_test)

rmse_train = root_mean_squared_error(y_tr, y_pred_train)
rmse_val = root_mean_squared_error(y_val, y_pred_val)
rmse_test = root_mean_squared_error(y_te, y_pred_test)

r2_train = r2_score(y_tr, y_pred_train)
r2_val = r2_score(y_val, y_pred_val)
r2_test = r2_score(y_te, y_pred_test)

In [189]:
mae_table.loc[2] = ('Lasso top10 MinMaxScaler', mae_train, mae_val, mae_test)
rmse_table.loc[2] = ('Lasso top10 MinMaxScaler', rmse_train, rmse_val, rmse_test)
r2_table.loc[2] = ('Lasso top10 MinMaxScaler', r2_train, r2_val, r2_test)

mae_table

,model,train,valid,test
1,Lasso MinMaxScaler,966.754012,1309.121111,1013.870440
2,Lasso top10 MinMaxScaler,794.196772,3635.423472,2770.156021


In [190]:
rmse_table

,model,train,valid,test
1,Lasso MinMaxScaler,8976.283541,45172.618130,11697.333550
2,Lasso top10 MinMaxScaler,1375.537017,3635.701857,2827.128587


In [191]:
r2_table

,model,train,valid,test
1,Lasso MinMaxScaler,0.035800,0.001657,0.022691
2,Lasso top10 MinMaxScaler,0.605172,-214.787417,-15.311543


In [192]:
def my_feature_selection(X, nan_procent=0.3, corr_procent=0.9):
    n_samples = X.shape[0]
    nan_ratio = np.isnan(X).sum(axis=0) / n_samples
    mask = nan_ratio < nan_procent
    X_filtered = X[:, mask]

    corr_matrix = np.corrcoef(X_filtered, rowvar=False)
    to_remove = set()
    for i in range(corr_matrix.shape[0]):
        for j in range(i+1, corr_matrix.shape[1]):
            if np.abs(corr_matrix[i,j]) > corr_procent:
                to_remove.add(j)

    ind = [i for i in range(X_filtered.shape[1]) if i not in to_remove]
    return X_filtered[:, ind]

In [193]:
X_filtered = my_feature_selection(X, nan_procent=0.3, corr_procent=0.9)
X10_fs = X_filtered[:, :10]
X_tr, X_val, X_te, y_tr, y_val, y_te = my_train_val_test_split(
    X10_fs, y, 0.2, 0.2, seed=21
)
scaler = MinMaxScaler()
X_tr = scaler.fit_transform(X_tr)
X_val = scaler.transform(X_val)
X_te = scaler.transform(X_te)
lasso_fs = Lasso(alpha=0.01, max_iter=10000)
lasso_fs.fit(X_tr, y_tr)
y_pred_train = lasso_fs.predict(X_tr)
y_pred_val = lasso_fs.predict(X_val)
y_pred_test = lasso_fs.predict(X_te)

mae_train = mean_absolute_error(y_tr, y_pred_train)
mae_val = mean_absolute_error(y_val, y_pred_val)
mae_test = mean_absolute_error(y_te, y_pred_test)

rmse_train = root_mean_squared_error(y_tr, y_pred_train)
rmse_val = root_mean_squared_error(y_val, y_pred_val)
rmse_test = root_mean_squared_error(y_te, y_pred_test)

r2_train = r2_score(y_tr, y_pred_train)
r2_val = r2_score(y_val, y_pred_val)
r2_test = r2_score(y_te, y_pred_test)
mae_table.loc[3] = ('Lasso feature selection MinMaxScaler', mae_train, mae_val, mae_test)
rmse_table.loc[3] = ('Lasso feature selection MinMaxScaler', rmse_train, rmse_val, rmse_test)
r2_table.loc[3] = ('Lasso feature selection MinMaxScaler', r2_train, r2_val, r2_test)
mae_table

,model,train,valid,test
1,Lasso MinMaxScaler,966.754012,1309.121111,1013.870440
2,Lasso top10 MinMaxScaler,794.196772,3635.423472,2770.156021
3,Lasso feature selection MinMaxScaler,962.742199,1300.054797,1007.780139


In [194]:
def permutation_importance(model, X, y, metric):
    score_before = metric(y, model.predict(X))
    importance = []
    for i in range(X.shape[1]):
        X_permuted = X.copy()
        np.random.shuffle(X_permuted[:, i])
        score = metric(y, model.predict(X_permuted))
        importance.append(score_before - score)
    return np.array(importance)

In [195]:
X_tr, X_val, X_te, y_tr, y_val, y_te = my_train_val_test_split(X, y, 0.2, 0.2, seed=21)
scaler = MinMaxScaler()
X_tr = scaler.fit_transform(X_tr)
X_val = scaler.transform(X_val)
X_te = scaler.transform(X_te)

lasso_perm = Lasso(alpha=0.01, max_iter=10000)
lasso_perm.fit(X_tr, y_tr)

importances = permutation_importance(lasso_perm, X_val, y_val, mean_absolute_error)
top10_idx = np.argsort(importances)[:10]
X_tr_top10 = X_tr[:, top10_idx]
X_val_top10 = X_val[:, top10_idx]
X_te_top10 = X_te[:, top10_idx]

lasso_perm_top10 = Lasso(alpha=0.01, max_iter=10000)
lasso_perm_top10.fit(X_tr_top10, y_tr)

y_pred_train = lasso_perm_top10.predict(X_tr_top10)
y_pred_val = lasso_perm_top10.predict(X_val_top10)
y_pred_test = lasso_perm_top10.predict(X_te_top10)

mae_train = mean_absolute_error(y_tr, y_pred_train)
mae_val = mean_absolute_error(y_val, y_pred_val)
mae_test = mean_absolute_error(y_te, y_pred_test)

rmse_train = root_mean_squared_error(y_tr, y_pred_train)
rmse_val = root_mean_squared_error(y_val, y_pred_val)
rmse_test = root_mean_squared_error(y_te, y_pred_test)

r2_train = r2_score(y_tr, y_pred_train)
r2_val = r2_score(y_val, y_pred_val)
r2_test = r2_score(y_te, y_pred_test)

mae_table.loc[4] = ('Lasso permutation MinMaxScaler', mae_train, mae_val, mae_test)
rmse_table.loc[4] = ('Lasso permutation MinMaxScaler', rmse_train, rmse_val, rmse_test)
r2_table.loc[4] = ('Lasso permutation MinMaxScaler', r2_train, r2_val, r2_test)

X1 = df[final_feature_list]
top10_features = X1.columns[top10_idx].tolist()
top10_df = pd.DataFrame({
        'Rank': range(1, 11),
        'Index': top10_idx,
        'Feature Name': X1.columns[top10_idx],
        'Importance': importances[top10_idx]
    })
    
print("\nТаблица топ-10 признаков:")
print(top10_df.to_string(index=False))


Таблица топ-10 признаков:
 Rank  Index      Feature Name  Importance
    1      0         bathrooms -453.707058
    2      7           Doorman -184.024192
    3      1          bedrooms -153.089600
    4      2    interest_level  -57.467969
    5     13     LaundryinUnit  -27.821588
    6     10 LaundryinBuilding  -14.080505
    7      6       DogsAllowed  -12.790790
    8      5       CatsAllowed  -11.829445
    9     21           Terrace   -4.245923
   10     17 HighSpeedInternet   -3.277615


In [196]:
import shap

lasso_shap = Lasso(alpha=0.01, max_iter=10000)
lasso_shap.fit(X_tr, y_tr)

explainer = shap.Explainer(lasso_shap, X_tr)
shap_values = explainer(X_val)

shap_importance = np.abs(shap_values.values).mean(axis=0)
top10_idx = np.argsort(shap_importance)[:10]

X_tr_top10 = X_tr[:, top10_idx]
X_val_top10 = X_val[:, top10_idx]
X_te_top10 = X_te[:, top10_idx]

lasso_shap_top10 = Lasso(alpha=0.01, max_iter=10000)
lasso_shap_top10.fit(X_tr_top10, y_tr)

y_pred_train = lasso_shap_top10.predict(X_tr_top10)
y_pred_val = lasso_shap_top10.predict(X_val_top10)
y_pred_test = lasso_shap_top10.predict(X_te_top10)

mae_train = mean_absolute_error(y_tr, y_pred_train)
mae_val = mean_absolute_error(y_val, y_pred_val)
mae_test = mean_absolute_error(y_te, y_pred_test)

rmse_train = root_mean_squared_error(y_tr, y_pred_train)
rmse_val = root_mean_squared_error(y_val, y_pred_val)
rmse_test = root_mean_squared_error(y_te, y_pred_test)

r2_train = r2_score(y_tr, y_pred_train)
r2_val = r2_score(y_val, y_pred_val)
r2_test = r2_score(y_te, y_pred_test)

mae_table.loc[5] = ('SHAP', mae_train, mae_val, mae_test)
rmse_table.loc[5] = ('SHAP', rmse_train, rmse_val, rmse_test)
r2_table.loc[5] = ('SHAP', r2_train, r2_val, r2_test)


In [197]:
mae_table

,model,train,valid,test
1,Lasso MinMaxScaler,966.754012,1309.121111,1013.870440
2,Lasso top10 MinMaxScaler,794.196772,3635.423472,2770.156021
3,Lasso feature selection MinMaxScaler,962.742199,1300.054797,1007.780139
4,Lasso permutation MinMaxScaler,953.088735,1294.870480,1001.811288
5,SHAP,1370.118060,1721.360919,1424.733434


In [198]:
rmse_table

,model,train,valid,test
1,Lasso MinMaxScaler,8976.283541,45172.618130,11697.333550
2,Lasso top10 MinMaxScaler,1375.537017,3635.701857,2827.128587
3,Lasso feature selection MinMaxScaler,8979.953052,45174.033682,11699.386097
4,Lasso permutation MinMaxScaler,8978.099705,45174.552625,11699.337753
5,SHAP,9123.066620,45210.857478,11817.822463


In [199]:
r2_table

,model,train,valid,test
1,Lasso MinMaxScaler,0.035800,0.001657,0.022691
2,Lasso top10 MinMaxScaler,0.605172,-214.787417,-15.311543
3,Lasso feature selection MinMaxScaler,0.035012,0.001595,0.022348
4,Lasso permutation MinMaxScaler,0.035410,0.001572,0.022356
5,SHAP,0.004009,-0.000034,0.002453


7. Оптимизация гиперпараметров

In [200]:
from sklearn.linear_model import ElasticNet
def my_grid_search(X_train, y_train, X_val, y_val, alphas, l1_ratios):
    best_score = -np.inf
    best_params = {}
    for alpha in alphas:
        for l1_ratio in l1_ratios:
            model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=21)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            score = r2_score(y_test, y_pred)
            
            if score > best_score:
                best_score = score
                best_params = {'alpha': alpha, 'l1_ratio': l1_ratio}
    
    return best_params, best_score

In [201]:
alphas = [0.001, 0.01, 0.1, 1]
l1_ratios = [0.1, 0.5, 0.9]

best_params_grid, best_score_grid = my_grid_search(X_train_scaled, y_train, 
                                                   X_val_scaled, y_val, 
                                                   alphas, l1_ratios)

print("Лучшие параметры для Grid Search:", best_params_grid, "R2:", best_score_grid)

model_grid = ElasticNet(**best_params_grid, max_iter=100007, random_state=21)
model_grid.fit(X_train_scaled, y_train)

y_pred_train_grid = model_grid.predict(X_train_scaled)
y_pred_val_grid = model_grid.predict(X_val_scaled)
y_pred_test_grid = model_grid.predict(X_test_scaled)

Лучшие параметры для Grid Search: {'alpha': 1, 'l1_ratio': 0.5} R2: 0.010098260700944994


In [202]:
def my_random_search(X_train, y_train, X_val, y_val, alphas, l1_ratios, iters=10):
    best_score = -np.inf
    best_params = {}
    for i in range(iters):
        alpha = np.random.uniform(*alphas)
        l1_ratio = np.random.uniform(*l1_ratios)
        model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=21)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        score = r2_score(y_val, y_pred)
        if score > best_score:
            best_params = {'alpha': alpha, 'l1_ratio': l1_ratio}

    return best_params, best_score

In [ ]:
best_params_rand, best_score_rand = my_random_search(X_train_scaled, y_train, 
                                                     X_val_scaled, y_val, 
                                                     (0.001, 1), (0.1, 0.9), iters=20)

print("Лучшие параметры для Random Search:", best_params_rand, "R2:", best_score_rand)


model_rand = ElasticNet(**best_params_rand, max_iter=10000, random_state=21)
model_rand.fit(X_train_scaled, y_train)

y_pred_train_rand = model_rand.predict(X_train_scaled)
y_pred_val_rand = model_rand.predict(X_val_scaled)
y_pred_test_rand = model_rand.predict(X_test_scaled)

Лучшие параметры для Random Search: {'alpha': 0.34457359348592753, 'l1_ratio': 0.8056604779562705} R2: -inf


In [213]:
import optuna

def objective(trial):
    try:
        alpha = trial.suggest_float('alpha', 0.001, 1, log=True)
        l1_ratio = trial.suggest_float('l1_ratio', 0.1, 0.9)
        model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000, random_state=21)
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_val_scaled)
        score = r2_score(y_val, y_pred)
        return score if np.isfinite(score) else -1e6
    except Exception as e:
        return -1e6 


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

best_params_optuna = study.best_params
best_score_optuna = study.best_value

print("Best Optuna:", best_params_optuna, "R2:", best_score_optuna)

model_optuna = ElasticNet(**best_params_optuna, max_iter=10000, random_state=21)
model_optuna.fit(X_train_scaled, y_train)

y_pred_train_opt = model_optuna.predict(X_train_scaled)
y_pred_val_opt = model_optuna.predict(X_val_scaled)
y_pred_test_opt = model_optuna.predict(X_test_scaled)


[I 2026-01-22 23:01:50,920] A new study created in memory with name: no-name-3e7ce49e-4ab0-4a8c-8140-db23104e97d3
[I 2026-01-22 23:01:51,182] Trial 0 finished with value: -1000000.0 and parameters: {'alpha': 0.07338766281643752, 'l1_ratio': 0.5440606815643227}. Best is trial 0 with value: -1000000.0.
[I 2026-01-22 23:01:51,270] Trial 1 finished with value: -1000000.0 and parameters: {'alpha': 0.004315861378849373, 'l1_ratio': 0.3316210033543958}. Best is trial 0 with value: -1000000.0.
[I 2026-01-22 23:01:51,320] Trial 2 finished with value: -1000000.0 and parameters: {'alpha': 0.032628461934241494, 'l1_ratio': 0.13585985308438292}. Best is trial 0 with value: -1000000.0.
[I 2026-01-22 23:01:51,405] Trial 3 finished with value: -1000000.0 and parameters: {'alpha': 0.0013250380175076819, 'l1_ratio': 0.3158708259045949}. Best is trial 0 with value: -1000000.0.
[I 2026-01-22 23:01:51,813] Trial 4 finished with value: -1000000.0 and parameters: {'alpha': 0.010656182621208744, 'l1_ratio': 0

Best Optuna: {'alpha': 0.07338766281643752, 'l1_ratio': 0.5440606815643227} R2: -1000000.0


In [205]:
mae_table.loc[6] = ('Grid Search', 
                    mean_absolute_error(y_train, y_pred_train_grid),
                    mean_absolute_error(y_val, y_pred_val_grid),
                    mean_absolute_error(y_test, y_pred_test_grid))

mae_table.loc[7] = ('Random Search',
                    mean_absolute_error(y_train, y_pred_train_rand),
                    mean_absolute_error(y_val, y_pred_val_rand),
                    mean_absolute_error(y_test, y_pred_test_rand))

mae_table.loc[8] = ('Optuna', 
                    mean_absolute_error(y_train, y_pred_train_opt),
                    mean_absolute_error(y_val, y_pred_val_opt),
                    mean_absolute_error(y_test, y_pred_test_opt))

rmse_table.loc[6] = ('Grid Search',
                     root_mean_squared_error(y_train, y_pred_train_grid),
                     root_mean_squared_error(y_val, y_pred_val_grid),
                     root_mean_squared_error(y_test, y_pred_test_grid))

rmse_table.loc[7] = ('Random Search',
                     root_mean_squared_error(y_train, y_pred_train_rand),
                     root_mean_squared_error(y_val, y_pred_val_rand),
                     root_mean_squared_error(y_test, y_pred_test_rand))

rmse_table.loc[8] = ('Optuna',
                     root_mean_squared_error(y_train, y_pred_train_opt),
                     root_mean_squared_error(y_val, y_pred_val_opt),
                     root_mean_squared_error(y_test, y_pred_test_opt))

r2_table.loc[6] = ('Grid Search', 
                   r2_score(y_train, y_pred_train_grid),
                   r2_score(y_val, y_pred_val_grid),
                   r2_score(y_test, y_pred_test_grid))

r2_table.loc[7] = ('Random Search', 
                   r2_score(y_train, y_pred_train_rand),
                   r2_score(y_val, y_pred_val_rand),
                   r2_score(y_test, y_pred_test_rand))

r2_table.loc[8] = ('Optuna', 
                   r2_score(y_train, y_pred_train_opt),
                   r2_score(y_val, y_pred_val_opt),
                   r2_score(y_test, y_pred_test_opt))


In [206]:
mae_table

,model,train,valid,test
1,Lasso MinMaxScaler,966.754012,1309.121111,1013.870440
2,Lasso top10 MinMaxScaler,794.196772,3635.423472,2770.156021
3,Lasso feature selection MinMaxScaler,962.742199,1300.054797,1007.780139
4,Lasso permutation MinMaxScaler,953.088735,1294.870480,1001.811288
5,SHAP,1370.118060,1721.360919,1424.733434
6,Grid Search,1312.658927,1666.827329,1368.418916
7,Random Search,1153.673151,1504.770138,1206.509095
8,Optuna,968.871940,1314.926955,1018.736030


In [207]:
rmse_table

,model,train,valid,test
1,Lasso MinMaxScaler,8976.283541,45172.618130,11697.333550
2,Lasso top10 MinMaxScaler,1375.537017,3635.701857,2827.128587
3,Lasso feature selection MinMaxScaler,8979.953052,45174.033682,11699.386097
4,Lasso permutation MinMaxScaler,8978.099705,45174.552625,11699.337753
5,SHAP,9123.066620,45210.857478,11817.822463
6,Grid Search,9107.287018,45202.352648,11804.324170
7,Random Search,9060.772077,45187.684125,11764.139360
8,Optuna,8986.410281,45170.134257,11702.158179


In [208]:
r2_table

,model,train,valid,test
1,Lasso MinMaxScaler,0.035800,0.001657,0.022691
2,Lasso top10 MinMaxScaler,0.605172,-214.787417,-15.311543
3,Lasso feature selection MinMaxScaler,0.035012,0.001595,0.022348
4,Lasso permutation MinMaxScaler,0.035410,0.001572,0.022356
5,SHAP,0.004009,-0.000034,0.002453
6,Grid Search,0.007451,0.000343,0.004731
7,Random Search,0.017564,0.000991,0.011496
8,Optuna,0.033623,0.001767,0.021884


In [211]:
folds = my_kfold(len(X_train_scaled), k=5, random_state=21)

best_params_list = []
scores_list = []

def objective(trial, X_tr, y_tr, X_val, y_val):
    alpha = trial.suggest_float('alpha', 0.001, 1, log=True)
    l1_ratio = trial.suggest_float('l1_ratio', 0.1, 0.9)
    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000, random_state=21)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val)
    
    if np.any(np.isnan(y_pred)) or np.any(np.isinf(y_pred)):
        return -1e6
    
    score = r2_score(y_val, y_pred)
    if not np.isfinite(score):
        return -1e6
    return score


for fold_num, (train_idx, val_idx) in enumerate(folds):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=21))
    study.optimize(lambda trial: objective(trial, X_tr, y_tr, X_val, y_val), n_trials=5)
    
    best_params_list.append(study.best_params)
    scores_list.append(study.best_value)
    
    print(f"Fold {fold_num} Best R2: {study.best_value:.6f}, params: {study.best_params}")

print("Mean R2:", np.mean(scores_list))


[I 2026-01-22 23:00:01,086] A new study created in memory with name: no-name-224b5059-103a-4967-ba78-b1b97ea90f1e


[I 2026-01-22 23:00:01,286] Trial 0 finished with value: 0.006662828445369806 and parameters: {'alpha': 0.0014001502504701327, 'l1_ratio': 0.33128772783185345}. Best is trial 0 with value: 0.006662828445369806.
[I 2026-01-22 23:00:01,339] Trial 1 finished with value: 0.006055863248656812 and parameters: {'alpha': 0.1455120772456994, 'l1_ratio': 0.11729299993275984}. Best is trial 0 with value: 0.006662828445369806.
[I 2026-01-22 23:00:01,950] Trial 2 finished with value: 0.0066559850840498536 and parameters: {'alpha': 0.0041473271596757285, 'l1_ratio': 0.14061860535630144}. Best is trial 0 with value: 0.006662828445369806.
[I 2026-01-22 23:00:02,297] Trial 3 finished with value: 0.0066575367580322675 and parameters: {'alpha': 0.008068925058345685, 'l1_ratio': 0.63112823569976}. Best is trial 0 with value: 0.006662828445369806.
[I 2026-01-22 23:00:02,518] Trial 4 finished with value: 0.006655756437520921 and parameters: {'alpha': 0.008401235903641736, 'l1_ratio': 0.566873020974879}. Bes

Fold 0 Best R2: 0.006663, params: {'alpha': 0.0014001502504701327, 'l1_ratio': 0.33128772783185345}


[I 2026-01-22 23:00:02,834] Trial 2 finished with value: 0.41986677006416784 and parameters: {'alpha': 0.0041473271596757285, 'l1_ratio': 0.14061860535630144}. Best is trial 2 with value: 0.41986677006416784.
[I 2026-01-22 23:00:02,952] Trial 3 finished with value: 0.4196020586741438 and parameters: {'alpha': 0.008068925058345685, 'l1_ratio': 0.63112823569976}. Best is trial 2 with value: 0.41986677006416784.
[I 2026-01-22 23:00:03,084] Trial 4 finished with value: 0.41990279981888734 and parameters: {'alpha': 0.008401235903641736, 'l1_ratio': 0.566873020974879}. Best is trial 4 with value: 0.41990279981888734.
[I 2026-01-22 23:00:03,095] A new study created in memory with name: no-name-2e102529-0d77-4e33-ba8c-8be14a823008
[I 2026-01-22 23:00:03,189] Trial 0 finished with value: 0.44373281184635016 and parameters: {'alpha': 0.0014001502504701327, 'l1_ratio': 0.33128772783185345}. Best is trial 0 with value: 0.44373281184635016.
[I 2026-01-22 23:00:03,213] Trial 1 finished with value: 0

Fold 1 Best R2: 0.419903, params: {'alpha': 0.008401235903641736, 'l1_ratio': 0.566873020974879}


...

Fold 2 Best R2: 0.444241, params: {'alpha': 0.008401235903641736, 'l1_ratio': 0.566873020974879}


[I 2026-01-22 23:00:03,826] Trial 2 finished with value: 0.4787235505319999 and parameters: {'alpha': 0.0041473271596757285, 'l1_ratio': 0.14061860535630144}. Best is trial 2 with value: 0.4787235505319999.
[I 2026-01-22 23:00:04,008] Trial 3 finished with value: 0.47873203408336085 and parameters: {'alpha': 0.008068925058345685, 'l1_ratio': 0.63112823569976}. Best is trial 3 with value: 0.47873203408336085.
[I 2026-01-22 23:00:04,101] Trial 4 finished with value: 0.47872367289165596 and parameters: {'alpha': 0.008401235903641736, 'l1_ratio': 0.566873020974879}. Best is trial 3 with value: 0.47873203408336085.
[I 2026-01-22 23:00:04,105] A new study created in memory with name: no-name-fd401e44-8951-447c-81df-fa80bdb2b3a0
[I 2026-01-22 23:00:04,164] Trial 0 finished with value: 0.45358762799522157 and parameters: {'alpha': 0.0014001502504701327, 'l1_ratio': 0.33128772783185345}. Best is trial 0 with value: 0.45358762799522157.
[I 2026-01-22 23:00:04,193] Trial 1 finished with value: 0.

Fold 3 Best R2: 0.478732, params: {'alpha': 0.008068925058345685, 'l1_ratio': 0.63112823569976}


[I 2026-01-22 23:00:04,396] Trial 3 finished with value: 0.45377909216570145 and parameters: {'alpha': 0.008068925058345685, 'l1_ratio': 0.63112823569976}. Best is trial 2 with value: 0.45381336888651047.
[I 2026-01-22 23:00:04,470] Trial 4 finished with value: 0.453819066119465 and parameters: {'alpha': 0.008401235903641736, 'l1_ratio': 0.566873020974879}. Best is trial 4 with value: 0.453819066119465.


Fold 4 Best R2: 0.453819, params: {'alpha': 0.008401235903641736, 'l1_ratio': 0.566873020974879}
Mean R2: 0.3606714767424953
